An apparently fast RAG service can return stale answers; an apparently successful request can hide a failed embedding call. This week adds two mechanisms with different purposes: caching avoids repeated work, while tracing makes work and failures explainable. Neither mechanism substitutes for answer-quality evaluation.

The reference is the [Week 6 release](https://github.com/jamwithai/production-agentic-rag-course/tree/week6.0), especially `src/services/cache/client.py`, `src/services/langfuse/`, and the updated ask endpoint. The learning goal is to define what makes two requests interchangeable, measure the latency distribution rather than a single speedup, and reconstruct a request from a trace.

## A cache key is an equivalence claim

Returning a cached answer asserts that recomputing this request would produce an acceptably equivalent result. The key must therefore represent every input relevant to that equivalence policy. Query text alone is insufficient if the answer also depends on model, corpus version, prompt, filters, generation parameters, or access rights.

The source hashes a canonical JSON object containing query, model, `top_k`, hybrid mode, and sorted categories. Sorting categories treats their order as irrelevant while preserving exact query text. That is a defensible normalization if category order truly has no semantic effect. Lowercasing or stripping punctuation from all queries is more dangerous: code, identifiers, and mathematical notation can distinguish those forms.

The source key does not include corpus revision, prompt revision, or tenant/permission context. The current public-paper application may not have tenants, but adding them later changes the key's security contract. A shared answer must never cross a permission boundary merely because two people asked the same words.

Hashing makes keys compact; it does not supply missing semantics. Likewise, a version field helps only if it changes when the corresponding input changes. A corpus-generation identifier, prompt hash, embedding/index generation, and authorization scope can support explicit invalidation policies. Tests of which changes produce new keys make that invalidation policy explicit.

**Experiment: expose a stale-equivalence bug.** The first key models the source's request-level inputs. The extended key adds a selected computation version. These are instructional implementations rather than imports from the project.


In [ ]:
import hashlib
import json

def cache_key(request, computation_version=None):
    data = {"query": request["query"], "model": request["model"],
            "top_k": request["top_k"], "use_hybrid": request["use_hybrid"],
            "categories": sorted(request.get("categories", []))}
    if computation_version is not None:
        data["computation_version"] = computation_version
    encoded = json.dumps(data, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()

request = {"query": "What is retrieval?", "model": "local-v1", "top_k": 3,
           "use_hybrid": True, "categories": ["cs.LG", "cs.AI"]}
reordered = dict(request, categories=["cs.AI", "cs.LG"])
assert cache_key(request) == cache_key(reordered)
assert cache_key(request, "corpus1/prompt1") != cache_key(request, "corpus2/prompt1")
print("Category ordering is canonical:", cache_key(request) == cache_key(reordered))
print("Different corpus generations require different keys:",
      cache_key(request, "corpus1/prompt1") != cache_key(request, "corpus2/prompt1"))


Reference output from isolated Python execution:

```text
Category ordering is canonical: True
Different corpus generations require different keys: True
```

The assertions establish properties of the key function. They do not establish freshness: the caller still has to provide the correct version and ensure concurrent requests see a coherent generation. Model aliases that change underneath a fixed string can also invalidate the equivalence assumption. Reproducibility consequently depends on the resolved model identity as well as its configured alias.

## TTL bounds retention, not truth

A time-to-live removes an entry after a duration. If a document changes one second after caching and TTL is an hour, the old answer may remain eligible for almost an hour. TTL therefore bounds one form of staleness; it does not invalidate immediately on data change.

Redis documents expiration semantics in [`EXPIRE`](https://redis.io/docs/latest/commands/expire/). The application still chooses its own policy for evidence updates, deletions, and permissions. Security revocations may require immediate invalidation rather than waiting for a freshness budget designed for public content.

**Experiment: use a fake clock to test the exact boundary.** A fake clock makes expiry tests deterministic without wall-clock waits. Here an entry is ineligible when the current time is equal to its deadline. A monotonic clock is appropriate for elapsed durations within a process; an external cache provides its own expiration mechanism.


In [ ]:
entries = {}

def store(key, value, now, ttl):
    entries[key] = (now + ttl, value)

def lookup(key, now):
    item = entries.get(key)
    if item is None:
        return None
    expires_at, value = item
    if now >= expires_at:
        del entries[key]
        return None
    return value

store("question", "answer from corpus v1", now=100, ttl=10)
assert lookup("question", now=109) == "answer from corpus v1"
assert lookup("question", now=110) is None
print("Before expiry: cached answer; at expiry: miss")


Reference output from isolated Python execution:

```text
Before expiry: cached answer; at expiry: miss
```

This test is deterministic because it controls time. It intentionally does not simulate Redis eviction under memory pressure, multiple clients, or failures. Those belong in the integration lab. A miss, an expired entry, an unavailable cache, and a corrupt cached payload can all invoke the normal answer pipeline, but they represent different operational conditions.

Repeated misses for one key can cause a **cache stampede**: many identical generations run at once before the first result is stored. Request coalescing or a bounded lock can reduce duplicate work, but it introduces waiting, ownership, and timeout decisions. Caching an error as an ordinary answer can preserve a transient failure after the dependency has recovered. Short-lived negative caching can be useful, but its status and invalidation rules should be explicit.

## The latency distribution users experience

With hit probability $h$, an illustrative mean latency is $\mathbb{E}[T]=hT_{\mathrm{hit}}+(1-h)T_{\mathrm{miss}}$. A high hit rate can produce an excellent mean while misses still dominate the tail. The workload's repeated-query distribution determines whether a cache is useful; replaying the same question repeatedly overstates its benefit for diverse questions.

**Experiment: a fast majority can hide a slow tail.** The timings are synthetic constants. The nearest-rank percentile definition is explicit so the output is reproducible.


In [ ]:
import math
from statistics import mean

latencies_ms = [20] * 90 + [2000] * 10

def nearest_rank(values, percentile):
    ordered = sorted(values)
    return ordered[math.ceil(percentile * len(ordered)) - 1]

print("Mean:", mean(latencies_ms), "ms")
print("p50:", nearest_rank(latencies_ms, 0.50), "ms")
print("p95:", nearest_rank(latencies_ms, 0.95), "ms")
assert mean(latencies_ms) == 218
assert nearest_rank(latencies_ms, 0.95) == 2000


Reference output from isolated Python execution:

```text
Mean: 218 ms
p50: 20 ms
p95: 2000 ms
```

A 20 ms median does not describe the experience of an uncached request. Separate hit and miss distributions, error rates, and workload composition make the aggregate latency interpretable. An average of individual request speedup ratios is also not interchangeable with total workload throughput. Cold startup, warm inference, and cache hits are distinct operating states with different resource costs.

Capacity can become more fragile after a cache is added. If the model server is provisioned only for the usual miss rate, a cache outage can suddenly expose it to the entire arrival rate. Admission control and bounded queues should prevent a helpful optimization from becoming a dependency whose failure overloads the core service.

## A trace should explain the outcome

A trace groups the operations of one request. Spans or observations record stages such as cache lookup, embeddings, retrieval, prompt construction, and generation. [Langfuse's data model](https://langfuse.com/docs/observability/data-model) describes how observations belong to a trace; the source wraps those concepts around the RAG pipeline.

An explainable request record includes request ID, corpus/index generation, prompt/model version, chosen retrieval mode, selected source IDs, fallback status, and stage durations. A request asking for hybrid retrieval can actually execute BM25 after an embedding failure. Reporting only the requested mode conceals the intervention that produced the answer.

Nested spans are not independent elapsed times to sum. A parent includes its children; concurrent siblings overlap. The critical path captures elapsed work without counting overlapping time twice. A monotonic clock measures durations, while wall-clock timestamps support correlation across processes; consistent units are necessary for both.

The source returns an entire stored response on a cache hit. Some fields may consequently describe the original computation rather than this request's service time. Response metadata can distinguish the original computation from the current delivery. A current trace ID should not accidentally identify another user's earlier request.

Traces can contain queries and excerpts. Collecting more data is not automatically better engineering. Retention, redaction, and access policies constrain what a trace can safely contain, particularly when an observability provider receives queries or documents. This is a design constraint on the tracing payload, not an excuse to omit operational visibility.

## Docker lab: prove cache semantics and trace completeness

1. Prepare `week6.0` with project name `rag-week6`, start its entire Compose stack, ingest data, and load the chosen Ollama model. Configure Langfuse using the release's `.env.example` and the local instance's project keys. Later releases change the Langfuse deployment and SDK; do not mix a Week 7 tracing example into this environment without a deliberate migration.

    ```bash
    docker compose --project-name rag-week6 up --build -d
    docker compose --project-name rag-week6 config --services
    docker compose --project-name rag-week6 exec redis redis-cli PING
    docker compose --project-name rag-week6 exec ollama ollama pull llama3.2:1b
    docker compose --project-name rag-week6 exec airflow \
      airflow dags trigger arxiv_paper_ingestion
    curl --fail-with-body --output /tmp/rag-week6-answer.json \
      --write-out 'HTTP %{http_code}; total %{time_total}s\n' \
      -H 'Content-Type: application/json' http://localhost:8000/api/v1/ask \
      -d '{"query":"How do the indexed papers use retrieval?","top_k":3,"use_hybrid":true,"model":"llama3.2:1b"}'
    docker compose --project-name rag-week6 exec redis \
      redis-cli --scan --pattern 'exact_cache:*'
    ```

2. Repeat the identical request and inspect the trace, Redis keys, response, and duration. Anticipate a generation on the first miss and a cache lookup without new generation on a hit, but confirm those stages actually occur. A faster second request can also result from warm model state, so latency alone does not prove caching. Inspect the TTL of the specific discovered key with `redis-cli TTL KEY`, replacing `KEY` with that exact key.

3. Change only `top_k`, then only categories, then the model. Predict whether each should miss before testing. Next, update an evidence document or prompt without changing the request. Under the source key, the old answer may still be returned. Record the stale-answer example and propose a versioned key or invalidation mechanism.

4. Stop the application Redis service temporarily, send one request, then restore it. The intended degraded behavior is a normal computation with explicit cache-error telemetry and bounded latency. It must not report a cache failure as evidence that the corpus is empty. Confirm restoration without flushing unrelated keys. Do not use `FLUSHALL` as a benchmark reset.

5. In Langfuse, inspect one miss, one hit, one embedding fallback, and one failed generation. Every outcome needs an understandable trace even when a stage never ran. Check that secrets are absent. Finish with a representative mixture of repeated and unique questions, separating cache-disabled baseline, warm misses, and hits. Retain p50, p95, failure rate, hit rate, and hardware/resource observations.

## Engineering review: optimize a service contract

1. Choose a freshness requirement: for example, newly ingested public papers should affect answers within a stated interval. Derive a compatible invalidation strategy and a test for it. Then explain why permission changes may require a stronger rule than ordinary corpus updates.

2. Review the source cache client: its async methods call synchronous Redis operations. Identify where blocking could occur and propose either an asynchronous client or controlled offloading, with timeouts and bounded concurrency. Changing syntax alone does not establish capacity.

3. Write a trace-based incident explanation for “hybrid requests got faster but less relevant.” Consider cache hits, silent lexical fallback, a changed corpus, and truncated context before blaming the model. A good explanation names the evidence that would separate those causes. That habit becomes essential once Week 7 adds adaptive execution paths.

## Sources and scope

- [Week 6 source](https://github.com/jamwithai/production-agentic-rag-course/tree/week6.0): exact-match key generation, TTL, response reuse, and tracing integration.
- [Redis expiration](https://redis.io/docs/latest/commands/expire/): expiration behavior; application freshness remains a separate policy.
- [Langfuse observability data model](https://langfuse.com/docs/observability/data-model): requests and observations, with SDK calls checked against the release being used.

All numerical latency examples are synthetic. The experiments establish key, expiry, and distribution properties; the Docker lab is required to measure actual caching, tracing, recovery, and throughput.
